In [1]:
import numpy as np
import pandas as pd
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns
import multiprocessing
import optuna
import warnings
from dateutil.relativedelta import relativedelta

# Sklearn imports
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, roc_auc_score
from sklearn.decomposition import TruncatedSVD, PCA, LatentDirichletAllocation
from sklearn.manifold import TSNE
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from category_encoders import TargetEncoder
from catboost import Pool
import catboost as ctb
import xgboost as xgb

# NLTK imports
import nltk
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords

# Ignore warnings
warnings.filterwarnings('ignore')


In [2]:
train2 = pd.read_csv('train_feature_v1.csv')
train2 = train2.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))
test2 = pd.read_csv('test_feature_v1.csv')
test2 = test2.rename(columns = lambda x:re.sub('[^A-Za-z0-9_]+', '', x))

In [3]:
pca_cols_train = train2.iloc[:,5:19]

le = LabelEncoder()
pca_cols_train = pca_cols_train.apply(le.fit_transform)

pca = PCA(n_components=5)  # Specify the number of components
pca_result = pca.fit_transform(pca_cols_train)
pca_result = pd.DataFrame(pca_result, columns=[f'pca_{i}' for i in range(5)])

train2 = train2.drop(columns=pca_cols_train.columns)
train2 = pd.concat([train2, pca_result], axis=1)

pca_cols_test = test2.iloc[:,5:19]

le = LabelEncoder()
pca_cols_test = pca_cols_test.apply(le.fit_transform)

pca = PCA(n_components=5)  # Specify the number of components
pca_result = pca.fit_transform(pca_cols_test)
pca_result = pd.DataFrame(pca_result, columns=[f'pca_{i}' for i in range(5)])

test2 = test2.drop(columns=pca_cols_train.columns)
test2 = pd.concat([test2, pca_result], axis=1)

In [4]:
pca_cols_train = train2.iloc[:,7:13]

le = LabelEncoder()
pca_cols_train = pca_cols_train.apply(le.fit_transform)

pca = PCA(n_components=3)  # Specify the number of components
pca_result = pca.fit_transform(pca_cols_train)
pca_result = pd.DataFrame(pca_result, columns=[f'pca_{i}_double' for i in range(3)])

train2 = train2.drop(columns=pca_cols_train.columns)
train2 = pd.concat([train2, pca_result], axis=1)

pca_cols_test = test2.iloc[:,7:13]

le = LabelEncoder()
pca_cols_test = pca_cols_test.apply(le.fit_transform)

pca = PCA(n_components=3)  # Specify the number of components
pca_result = pca.fit_transform(pca_cols_test)
pca_result = pd.DataFrame(pca_result, columns=[f'pca_{i}_double' for i in range(3)])

test2 = test2.drop(columns=pca_cols_train.columns)
test2 = pd.concat([test2, pca_result], axis=1)

In [5]:
X_train = train2.drop(columns=['price'])
Y_train = train2[['price']]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X_train, Y_train, test_size=0.2, random_state=42)

In [7]:
cat_model = ctb.CatBoostRegressor(iterations=1000, learning_rate=0.1, depth=5, random_seed=42, verbose=100)

# Fit the model
cat_model.fit(X_train, y_train, eval_set=(X_test, y_test))

0:	learn: 145907.9821443	test: 132523.9626288	best: 132523.9626288 (0)	total: 141ms	remaining: 2m 21s
100:	learn: 20845.6403761	test: 26019.1827353	best: 26019.1827353 (100)	total: 339ms	remaining: 3.02s
200:	learn: 14553.5008285	test: 23945.7600688	best: 23945.7600688 (200)	total: 535ms	remaining: 2.13s
300:	learn: 11304.3879280	test: 23128.1831564	best: 23128.1831564 (300)	total: 745ms	remaining: 1.73s
400:	learn: 9287.0927901	test: 22578.2073480	best: 22565.4292492 (399)	total: 959ms	remaining: 1.43s
500:	learn: 7851.3758151	test: 22295.0581856	best: 22292.7865503 (499)	total: 1.2s	remaining: 1.2s
600:	learn: 6893.4268442	test: 22178.3417321	best: 22163.5491984 (563)	total: 1.44s	remaining: 959ms
700:	learn: 6189.8939726	test: 22047.2288892	best: 22044.3346147 (699)	total: 1.67s	remaining: 712ms
800:	learn: 5557.9764696	test: 21903.4671594	best: 21902.4856012 (798)	total: 1.87s	remaining: 465ms
900:	learn: 5087.6532172	test: 21782.6642801	best: 21782.6642801 (900)	total: 2.07s	remai

In [8]:
train_label_encoded = pd.read_csv('train_feature_label_encoded.csv')
test_label_encoded = pd.read_csv('test_feature_label_encoded.csv')

# Separate features and target
X = train_label_encoded[['feature_make_imputed']]
y = train_label_encoded['price']

X_train_label_encoded = train_label_encoded.drop(columns=['price'])
Y_train_label_encoded = train_label_encoded[['price']]

X_train_label_encoded, X_test_label_encoded, y_train_label_encoded, y_test_label_encoded = train_test_split(X_train_label_encoded, Y_train_label_encoded, test_size=0.2, random_state=42)

In [9]:
categorical_features = ["feature_is_electric", "feature_make_imputed", "feature_fuel_label", "feature_low_mileage", "feature_type", "feature_ownership", "feature_condition", "feature_special_features", "feature_sales", "feature_price_category", "feature_type_of_vehicle", ]

In [10]:
cat_model_label_encoded = ctb.CatBoostRegressor(iterations=1000, learning_rate=0.1, depth=5, random_seed=42, verbose=100)

# Fit the model
cat_model_label_encoded.fit(X_train_label_encoded, y_train_label_encoded, eval_set=(X_test_label_encoded, y_test_label_encoded),cat_features=categorical_features)

0:	learn: 145758.1192296	test: 132561.4131600	best: 132561.4131600 (0)	total: 21.8ms	remaining: 21.8s
100:	learn: 21716.7410326	test: 26937.0886895	best: 26937.0886895 (100)	total: 2.12s	remaining: 18.8s
200:	learn: 15637.5434018	test: 24815.0799788	best: 24812.1245557 (199)	total: 4.07s	remaining: 16.2s
300:	learn: 12430.9054712	test: 23901.0606201	best: 23869.3261191 (295)	total: 6.05s	remaining: 14s
400:	learn: 10258.7266512	test: 23413.4588555	best: 23405.0581214 (395)	total: 7.93s	remaining: 11.8s
500:	learn: 8805.0855040	test: 23210.6110764	best: 23198.1213048 (496)	total: 9.82s	remaining: 9.78s
600:	learn: 7701.8789735	test: 23021.1686760	best: 22997.5265125 (594)	total: 11.7s	remaining: 7.77s
700:	learn: 6948.0980776	test: 22874.2120612	best: 22867.7999202 (694)	total: 13.6s	remaining: 5.79s
800:	learn: 6328.3563933	test: 22846.1984863	best: 22833.2044456 (784)	total: 15.4s	remaining: 3.83s
900:	learn: 5823.4889412	test: 22760.1714497	best: 22760.1714497 (900)	total: 17.3s	rema

In [11]:
# GridSearch Optimization
param_grid = {
    'iterations': [500, 1000, 1500],
    'learning_rate': [0.05, 0.1, 0.15],
    'depth': [3, 5, 7],
    'random_seed': [42]
}

grid_search = GridSearchCV(
    estimator=cat_model_label_encoded,
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    verbose=1,
)

grid_search.fit(X_train_label_encoded, y_train_label_encoded, cat_features=categorical_features)
print("GridSearch Best Parameters:")
print(grid_search.best_params_)

# Get the best model from GridSearch
grid_best_model = grid_search.best_estimator_


Fitting 5 folds for each of 27 candidates, totalling 135 fits
0:	learn: 152236.0967069	total: 8.49ms	remaining: 4.24s
100:	learn: 34324.8376670	total: 1.03s	remaining: 4.09s
200:	learn: 27811.2188559	total: 2.15s	remaining: 3.19s
300:	learn: 24196.4110495	total: 3.16s	remaining: 2.09s
400:	learn: 21763.6180277	total: 4.18s	remaining: 1.03s
499:	learn: 19848.6667613	total: 5.21s	remaining: 0us
0:	learn: 151793.2448033	total: 9.65ms	remaining: 4.82s
100:	learn: 33735.6908374	total: 1.03s	remaining: 4.06s
200:	learn: 27466.6845601	total: 2.05s	remaining: 3.06s
300:	learn: 23612.2548344	total: 3.06s	remaining: 2.02s
400:	learn: 21389.3526926	total: 4.09s	remaining: 1.01s
499:	learn: 19790.6395098	total: 5.1s	remaining: 0us
0:	learn: 154908.2630289	total: 9.49ms	remaining: 4.74s
100:	learn: 34310.5067138	total: 1.03s	remaining: 4.08s
200:	learn: 27919.8941572	total: 2.05s	remaining: 3.05s
300:	learn: 24399.3009909	total: 3.06s	remaining: 2.02s
400:	learn: 21897.8566106	total: 4.08s	remainin

In [12]:
# Optuna Optimization
def objective(trial):
    iterations = trial.suggest_int('iterations', 500, 1500)
    learning_rate = trial.suggest_float('learning_rate', 0.05, 0.15)
    depth = trial.suggest_int('depth', 3, 7)
    
    model = ctb.CatBoostRegressor(
        iterations=iterations,
        learning_rate=learning_rate,
        depth=depth,
        random_seed=42,
        verbose=0
    )
    
    model.fit(X_train_label_encoded, y_train_label_encoded, cat_features=categorical_features)
    rmse = np.sqrt(mean_squared_error(y_test_label_encoded, model.predict(X_test_label_encoded)))
    return rmse

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50)

print("Optuna Best Parameters:")
print(study.best_params)

# Get the best model from Optuna
optuna_best_model = ctb.CatBoostRegressor(**study.best_params, random_seed=42)
optuna_best_model.fit(X_train_label_encoded, y_train_label_encoded, cat_features=categorical_features)

[I 2024-11-14 11:48:34,764] A new study created in memory with name: no-name-648a9cd5-e523-4e3f-bc12-df85bd6bc01f
[I 2024-11-14 11:48:54,270] Trial 0 finished with value: 22079.583985720532 and parameters: {'iterations': 776, 'learning_rate': 0.0518555836569665, 'depth': 7}. Best is trial 0 with value: 22079.583985720532.
[I 2024-11-14 11:49:12,653] Trial 1 finished with value: 22681.879997832686 and parameters: {'iterations': 888, 'learning_rate': 0.12178601339350587, 'depth': 6}. Best is trial 0 with value: 22079.583985720532.
[I 2024-11-14 11:49:41,363] Trial 2 finished with value: 22647.3647507298 and parameters: {'iterations': 1391, 'learning_rate': 0.07695811491778096, 'depth': 6}. Best is trial 0 with value: 22079.583985720532.
[I 2024-11-14 11:49:56,089] Trial 3 finished with value: 23097.5094396357 and parameters: {'iterations': 712, 'learning_rate': 0.05671179082401422, 'depth': 6}. Best is trial 0 with value: 22079.583985720532.
[I 2024-11-14 11:50:09,853] Trial 4 finished w

Optuna Best Parameters:
{'iterations': 920, 'learning_rate': 0.11131826071894371, 'depth': 7}
0:	learn: 143624.8630628	total: 25.2ms	remaining: 23.1s
1:	learn: 131208.7441428	total: 52.7ms	remaining: 24.2s
2:	learn: 119718.8765452	total: 79.7ms	remaining: 24.4s
3:	learn: 110064.1194960	total: 106ms	remaining: 24.3s
4:	learn: 100984.7595676	total: 135ms	remaining: 24.7s
5:	learn: 92877.0169225	total: 164ms	remaining: 25s
6:	learn: 85771.6291409	total: 191ms	remaining: 24.9s
7:	learn: 79243.5723710	total: 218ms	remaining: 24.9s
8:	learn: 73245.1522969	total: 245ms	remaining: 24.8s
9:	learn: 68085.4355158	total: 273ms	remaining: 24.8s
10:	learn: 63595.3543454	total: 299ms	remaining: 24.7s
11:	learn: 59613.5824974	total: 327ms	remaining: 24.7s
12:	learn: 55738.2737721	total: 353ms	remaining: 24.6s
13:	learn: 52306.4824133	total: 378ms	remaining: 24.5s
14:	learn: 49520.5914391	total: 402ms	remaining: 24.3s
15:	learn: 46922.8750033	total: 429ms	remaining: 24.2s
16:	learn: 44461.4808307	total

In [13]:
# Get the best model from Optuna
optuna_best_model = ctb.CatBoostRegressor(**study.best_params, random_seed=42)
optuna_best_model.fit(X_train_label_encoded, y_train_label_encoded, cat_features=categorical_features)

0:	learn: 143624.8630628	total: 23.3ms	remaining: 21.5s
1:	learn: 131208.7441428	total: 48.1ms	remaining: 22.1s
2:	learn: 119718.8765452	total: 73ms	remaining: 22.3s
3:	learn: 110064.1194960	total: 99ms	remaining: 22.7s
4:	learn: 100984.7595676	total: 125ms	remaining: 22.8s
5:	learn: 92877.0169225	total: 150ms	remaining: 22.9s
6:	learn: 85771.6291409	total: 177ms	remaining: 23.1s
7:	learn: 79243.5723710	total: 203ms	remaining: 23.1s
8:	learn: 73245.1522969	total: 236ms	remaining: 23.9s
9:	learn: 68085.4355158	total: 262ms	remaining: 23.8s
10:	learn: 63595.3543454	total: 288ms	remaining: 23.8s
11:	learn: 59613.5824974	total: 317ms	remaining: 24s
12:	learn: 55738.2737721	total: 343ms	remaining: 24s
13:	learn: 52306.4824133	total: 371ms	remaining: 24s
14:	learn: 49520.5914391	total: 398ms	remaining: 24s
15:	learn: 46922.8750033	total: 424ms	remaining: 24s
16:	learn: 44461.4808307	total: 450ms	remaining: 23.9s
17:	learn: 42546.9658035	total: 477ms	remaining: 23.9s
18:	learn: 40662.4192189	

In [14]:
# Evaluate the GridSearch Best Model
print("GridSearch Best Model:")
grid_train_rmse = mean_squared_error(y_train_label_encoded, grid_best_model.predict(X_train_label_encoded), squared=False)
grid_test_rmse = mean_squared_error(y_test_label_encoded, grid_best_model.predict(X_test_label_encoded), squared=False)
grid_train_mae = mean_absolute_error(y_train_label_encoded, grid_best_model.predict(X_train_label_encoded))
grid_test_mae = mean_absolute_error(y_test_label_encoded, grid_best_model.predict(X_test_label_encoded))
grid_train_r2 = r2_score(y_train_label_encoded, grid_best_model.predict(X_train_label_encoded))
grid_test_r2 = r2_score(y_test_label_encoded, grid_best_model.predict(X_test_label_encoded))
print(f"Train RMSE: {grid_train_rmse:.2f}, Test RMSE: {grid_test_rmse:.2f}")
print(f"Train MAE: {grid_train_mae:.2f}, Test MAE: {grid_test_mae:.2f}")
print(f"Train R-squared: {grid_train_r2:.2f}, Test R-squared: {grid_test_r2:.2f}")

# Evaluate the Optuna Best Model
print("\nOptuna Best Model:")
optuna_train_rmse = mean_squared_error(y_train_label_encoded, optuna_best_model.predict(X_train_label_encoded), squared=False)
optuna_test_rmse = mean_squared_error(y_test_label_encoded, optuna_best_model.predict(X_test_label_encoded), squared=False)
optuna_train_mae = mean_absolute_error(y_train_label_encoded, optuna_best_model.predict(X_train_label_encoded))
optuna_test_mae = mean_absolute_error(y_test_label_encoded, optuna_best_model.predict(X_test_label_encoded))
optuna_train_r2 = r2_score(y_train_label_encoded, optuna_best_model.predict(X_train_label_encoded))
optuna_test_r2 = r2_score(y_test_label_encoded, optuna_best_model.predict(X_test_label_encoded))
print(f"Train RMSE: {optuna_train_rmse:.2f}, Test RMSE: {optuna_test_rmse:.2f}")
print(f"Train MAE: {optuna_train_mae:.2f}, Test MAE: {optuna_test_mae:.2f}")
print(f"Train R-squared: {optuna_train_r2:.2f}, Test R-squared: {optuna_test_r2:.2f}")

# Evaluate the CatBoost Model
print("\nCatBoost Model:")
cat_model_train_rmse = mean_squared_error(y_train, cat_model.predict(X_train), squared=False)
cat_model_test_rmse = mean_squared_error(y_test, cat_model.predict(X_test), squared=False)
cat_model_train_mae = mean_absolute_error(y_train, cat_model.predict(X_train))
cat_model_test_mae = mean_absolute_error(y_test, cat_model.predict(X_test))
cat_model_train_r2 = r2_score(y_train, cat_model.predict(X_train))
cat_model_test_r2 = r2_score(y_test, cat_model.predict(X_test))
print(f"Train RMSE: {cat_model_train_rmse:.2f}, Test RMSE: {cat_model_test_rmse:.2f}")
print(f"Train MAE: {cat_model_train_mae:.2f}, Test MAE: {cat_model_test_mae:.2f}")
print(f"Train R-squared: {cat_model_train_r2:.2f}, Test R-squared: {cat_model_test_r2:.2f}")

# Evaluate the CatBoost Label Encoded Model
print("\nCatBoost Label Encoded Model:")
cat_model_label_train_rmse = mean_squared_error(y_train_label_encoded, cat_model_label_encoded.predict(X_train_label_encoded), squared=False)
cat_model_label_test_rmse = mean_squared_error(y_test_label_encoded, cat_model_label_encoded.predict(X_test_label_encoded), squared=False)
cat_model_label_train_mae = mean_absolute_error(y_train_label_encoded, cat_model_label_encoded.predict(X_train_label_encoded))
cat_model_label_test_mae = mean_absolute_error(y_test_label_encoded, cat_model_label_encoded.predict(X_test_label_encoded))
cat_model_label_train_r2 = r2_score(y_train_label_encoded, cat_model_label_encoded.predict(X_train_label_encoded))
cat_model_label_test_r2 = r2_score(y_test_label_encoded, cat_model_label_encoded.predict(X_test_label_encoded))
print(f"Train RMSE: {cat_model_label_train_rmse:.2f}, Test RMSE: {cat_model_label_test_rmse:.2f}")
print(f"Train MAE: {cat_model_label_train_mae:.2f}, Test MAE: {cat_model_label_test_mae:.2f}")
print(f"Train R-squared: {cat_model_label_train_r2:.2f}, Test R-squared: {cat_model_label_test_r2:.2f}")

GridSearch Best Model:
Train RMSE: 4774.22, Test RMSE: 22472.08
Train MAE: 2519.12, Test MAE: 4565.78
Train R-squared: 1.00, Test R-squared: 0.98

Optuna Best Model:
Train RMSE: 4335.15, Test RMSE: 21905.63
Train MAE: 2213.80, Test MAE: 4374.64
Train R-squared: 1.00, Test R-squared: 0.98

CatBoost Model:
Train RMSE: 4717.88, Test RMSE: 21741.32
Train MAE: 2821.76, Test MAE: 4719.25
Train R-squared: 1.00, Test R-squared: 0.98

CatBoost Label Encoded Model:
Train RMSE: 6139.26, Test RMSE: 22660.82
Train MAE: 3177.34, Test MAE: 5004.94
Train R-squared: 1.00, Test R-squared: 0.98


In [15]:
# Make predictions for all 4 models
grid_preds = grid_best_model.predict(test_label_encoded)
optuna_preds = optuna_best_model.predict(test_label_encoded)
cat_model_preds = cat_model.predict(test2)
cat_model_label_preds = cat_model_label_encoded.predict(test_label_encoded)

# Export each model's predictions to a separate CSV file
pd.DataFrame({'Id': range(len(test_label_encoded)), 'Predicted': grid_preds}).to_csv('grid_search_predictions.csv', index=False)
pd.DataFrame({'Id': range(len(test_label_encoded)), 'Predicted': optuna_preds}).to_csv('optuna_predictions.csv', index=False)
pd.DataFrame({'Id': range(len(test2)), 'Predicted': cat_model_preds}).to_csv('cat_model_predictions.csv', index=False)
pd.DataFrame({'Id': range(len(test_label_encoded)), 'Predicted': cat_model_label_preds}).to_csv('cat_model_label_predictions.csv', index=False)

print("Predictions exported to separate CSV files.")

Predictions exported to separate CSV files.
